<a href="https://colab.research.google.com/github/quickklearner/Sequence_Marketing/blob/main/Sequence%20Marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DATA 6400 Final Project Robust Fake Review Detection with Large Language Models: A Prompt Rewording and Review Length Analysis**

## Jose Barral, Victoria González, Arjun Moitra & Ooreoluwa Ayoola
## April 10th, 2026

Acknowledgments:
* We used ChatGPT for some syntax guidance and all instances of this are commented appropriately below.
* All other programming, text explanations and analysis are our own.

## 1. Imports and Setup

First, we need to download an older version of numpy in order to be able to use the hugging face dataset.

In [ ]:
!pip -q uninstall -y datasets
!pip -q install datasets==2.18.0

!pip install -U "numpy<2.0" transformers accelerate evaluate

In [ ]:
!pip -q install -U sentencepiece

Libraries Needed:

In [ ]:
#Dataset Import
import datasets
from datasets import load_dataset

#Data Handling
import pandas as pd
import numpy as np

#Tokenization
from collections import Counter
from transformers import AutoTokenizer

#Parsing
import re

#Modeling
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, precision_score, recall_score, f1_score

#Google Drive (Saving of results)
from google.colab import drive

Then, we load the hugging face dataset.

In [ ]:
print(datasets.__version__)

#Load of hugging face dataset
ds = load_dataset("difraud/difraud", "product_reviews", trust_remote_code=True)
ds

2.18.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16776
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2097
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2098
    })
})

As shown, the dataset is already split into training, validation, and test sets, containing 16,776, 2,097, and 2,098 observations, respectively.

## 2. Data Quality Check

In [ ]:
train_df = pd.DataFrame(ds["train"])
val_df   = pd.DataFrame(ds["validation"])
test_df  = pd.DataFrame(ds["test"])

train_df.head()

,text,label
0,J. D Salinger was righteous... Not in his scri...,1
1,"Take my word, this is one of the best grill co...",1
2,"As other reviewers have mentioned, the packagi...",0
3,Great bookcase for small space and is sturdy e...,0
4,It's ok... I pirchased this thinking it'd male...,0


In [ ]:
# Any null values?
print(train_df.isnull().sum())

# Class balance?
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True))

# Basic length
train_df["text_length"] = train_df["text"].str.len()
print(train_df["text_length"].describe())

text     0
label    0
dtype: int64
label
1    8393
0    8383
Name: count, dtype: int64
label
1    0.500298
0    0.499702
Name: proportion, dtype: float64
count    16776.000000
mean       361.855269
std        460.182063
min         62.000000
25%        154.000000
50%        233.000000
75%        381.000000
max      15957.000000
Name: text_length, dtype: float64


The dataset shows no missing values in either the text or label variables, indicating that it is complete and ready for modeling.

In terms of class distribution, the dataset is perfectly balanced, with 8,393 instances labeled as 1 and 8,383 as 0. This corresponds to proportions of 50.03% and 49.97%, respectively, ensuring no class imbalance issues that could bias the model.

Regarding text length, there is significant variability across reviews. The average length is approximately 362 characters, with a median of 233, indicating a right-skewed distribution. The shortest review contains 62 characters, while the longest reaches 15,957 characters. The interquartile range (154 to 381 characters) shows that most reviews are relatively short, although the presence of very long outliers suggests that truncation during tokenization could be necessary.

## 3. Length Analysis

We analyze the token lengths to define a balanced categorization of reviews into short, medium, and long groups.

In [ ]:
#For this, we will use the Roberta base tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def count_tokens(example):
    tokens = tokenizer.tokenize(example["text"])
    example["n_tokens"] = len(tokens) + 2
    return example

ds = ds.map(count_tokens)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16776 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (593 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/2097 [00:00<?, ? examples/s]

Map:   0%|          | 0/2098 [00:00<?, ? examples/s]

In [ ]:
train_lengths = ds["train"]["n_tokens"]

q1 = np.quantile(train_lengths, 0.33)
q2 = np.quantile(train_lengths, 0.66)

print("Q1:", q1)
print("Q2:", q2)

Q1: 42.0
Q2: 71.0


Then, reviews are categorized as follows: short (≤ 42 tokens), medium (43–71 tokens), and long (> 71 tokens).

Then, we create the length groups based on the previously identified quantile thresholds.

In [ ]:
def assign_length_group(example):
    if example["n_tokens"] <= q1:
        example["length_group"] = "short"
    elif example["n_tokens"] <= q2:
        example["length_group"] = "medium"
    else:
        example["length_group"] = "long"
    return example

ds = ds.map(assign_length_group)

Map:   0%|          | 0/16776 [00:00<?, ? examples/s]

Map:   0%|          | 0/2097 [00:00<?, ? examples/s]

Map:   0%|          | 0/2098 [00:00<?, ? examples/s]

In [ ]:
print("Train:", Counter(ds["train"]["length_group"]))
print("Validation:", Counter(ds["validation"]["length_group"]))
print("Test:", Counter(ds["test"]["length_group"]))

Train: Counter({'long': 5666, 'short': 5622, 'medium': 5488})
Validation: Counter({'long': 731, 'medium': 696, 'short': 670})
Test: Counter({'long': 722, 'short': 699, 'medium': 677})


## 4. Models

### 4.1. Roberta Model

We will use Roberta as the baseline model.

#### 4.1.1. Model architecture

We tokenize the input text using the RoBERTa tokenizer. Truncation and padding are applied to a maximum sequence length of 512 tokens to ensure uniform input size across all samples.

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

ds_tokenized = ds.map(tokenize, batched=True)

Map:   0%|          | 0/16776 [00:00<?, ? examples/s]

Map:   0%|          | 0/2097 [00:00<?, ? examples/s]

Map:   0%|          | 0/2098 [00:00<?, ? examples/s]

After tokenization, the original text column is removed, and the label column is renamed to match the expected input format (labels). The dataset is then converted to PyTorch tensors for model training.

In [ ]:
ds_tokenized = ds_tokenized.remove_columns(["text"])
ds_tokenized = ds_tokenized.rename_column("label", "labels")

ds_tokenized.set_format("torch")

A pre-trained RoBERTa-base model is loaded for sequence classification, with a binary output layer (num_labels = 2) corresponding to the FAKE/REAL classification task.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model performance is evaluated using accuracy, precision, recall, and F1-score. Predictions are obtained by selecting the class with the highest logit score. Given the nature of the task, recall is used as the main metric for model selection to prioritize the detection of fake reviews.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

Evaluation and model saving are performed at the end of each epoch, and the best model is selected based on recall. Weight decay is applied to improve generalization.

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True
)

Then, the Hugging Face Trainer is used to manage the training and evaluation process by integrating the model, training arguments, datasets, and evaluation metrics into a unified pipeline.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    eval_dataset=ds_tokenized["validation"],
    compute_metrics=compute_metrics
)

#### 4.1.2. Model training

In [ ]:
trainer.train()

{'loss': '0.6375', 'grad_norm': '6.986', 'learning_rate': '1.524e-05', 'epoch': '0.4766'}
{'loss': '0.5849', 'grad_norm': '16.68', 'learning_rate': '1.048e-05', 'epoch': '0.9533'}
{'eval_loss': '0.5762', 'eval_accuracy': '0.7067', 'eval_precision': '0.7754', 'eval_recall': '0.5825', 'eval_f1': '0.6652', 'eval_runtime': '61.25', 'eval_samples_per_second': '34.24', 'eval_steps_per_second': '2.155', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.5104', 'grad_norm': '9.82', 'learning_rate': '5.71e-06', 'epoch': '1.43'}
{'loss': '0.4967', 'grad_norm': '10.35', 'learning_rate': '9.438e-07', 'epoch': '1.907'}
{'eval_loss': '0.5743', 'eval_accuracy': '0.7225', 'eval_precision': '0.7132', 'eval_recall': '0.7445', 'eval_f1': '0.7285', 'eval_runtime': '61.39', 'eval_samples_per_second': '34.16', 'eval_steps_per_second': '2.15', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'train_runtime': '3407', 'train_samples_per_second': '9.848', 'train_steps_per_second': '0.616', 'train_loss': '0.5539', 'epoch': '2'}


TrainOutput(global_step=2098, training_loss=0.5538772186628401, metrics={'train_runtime': 3407.1581, 'train_samples_per_second': 9.848, 'train_steps_per_second': 0.616, 'train_loss': 0.5538772186628401, 'epoch': 2.0})

#### 4.1.3. Model evaluation

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=ds_tokenized["test"])
print(test_metrics)

{'eval_loss': '0.5696', 'eval_accuracy': '0.7193', 'eval_precision': '0.7006', 'eval_recall': '0.7667', 'eval_f1': '0.7322', 'eval_runtime': '62.33', 'eval_samples_per_second': '33.66', 'eval_steps_per_second': '2.118', 'epoch': '2'}
{'eval_loss': 0.5696370005607605, 'eval_accuracy': 0.719256434699714, 'eval_precision': 0.700609225413403, 'eval_recall': 0.7666666666666667, 'eval_f1': 0.7321509777171441, 'eval_runtime': 62.3305, 'eval_samples_per_second': 33.659, 'eval_steps_per_second': 2.118, 'epoch': 2.0}


In [ ]:
# Test predictions
pred_output = trainer.predict(ds_tokenized["test"])
pred_labels = pred_output.predictions.argmax(axis=1)
true_labels = pred_output.label_ids

# Unify with original dataset
test_results_df = pd.DataFrame(ds["test"])
test_results_df["pred_label"] = pred_labels
test_results_df["true_label"] = true_labels

# Metrics by group
group_metrics = []

for group in ["short", "medium", "long"]:
    subset = test_results_df[test_results_df["length_group"] == group]

    y_true = subset["true_label"]
    y_pred = subset["pred_label"]

    group_metrics.append({
        "length_group": group,
        "n": len(subset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    })

group_metrics_df = pd.DataFrame(group_metrics)
print(group_metrics_df)

  length_group    n  accuracy  precision    recall        f1
0        short  699  0.698140   0.644699  0.721154  0.680787
1       medium  677  0.704579   0.727848  0.829327  0.775281
2         long  722  0.753463   0.720859  0.729814  0.725309


Since the primary objective of this study is to maximize recall, we focus on the model’s ability to correctly identify deceptive reviews. The results show that recall varies across review length groups. Medium-length reviews achieved the highest recall (0.829), indicating that the model successfully identifies a large proportion of deceptive reviews in this category. In comparison, recall decreases for long reviews (0.729) and short reviews (0.721). This suggests that reviews of moderate length provide the most informative signals for detecting deception, while very short reviews lack sufficient context and very long reviews may introduce additional noise.

#### 4.1.4. Saving of results

In [ ]:
drive.mount('/content/drive')

In [ ]:
#Final metrics of model
group_metrics_df.to_csv(
"/content/drive/MyDrive/roberta_metrics_by_length.csv",
index=False
)

In [ ]:
#Test prediction
test_results_df.to_csv(
"/content/drive/MyDrive/roberta_test_predictions.csv",
index=False
)

In [ ]:
#Trained model
trainer.save_model("/content/drive/MyDrive/roberta_fake_review_model")

#If we need it, just run:
#from transformers import AutoModelForSequenceClassification

#model = AutoModelForSequenceClassification.from_pretrained(
#"/content/drive/MyDrive/roberta_fake_review_model"
#)

### 4.2. LLM Zero-Shot Model (Mistral)

#### 4.2.1. Model setup

We use the Mistral-7B-Instruct model for zero-shot classification. The model is loaded using the Hugging Face Transformers library, along with its corresponding tokenizer. To optimize performance, half-precision (float16) is used when a GPU is available, and the model is automatically distributed across devices using device_map="auto".

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

#### 4.2.1. Prompt design

The task is framed as a zero-shot classification problem, where each review is embedded into a prompt instructing the model to classify it as FAKE or REAL. Different prompt variations are tested to evaluate their impact on performance.

In [ ]:
prompt_A = """Classify this product review as FAKE or REAL.

A REAL review usually reflects genuine product experience, including concrete details, natural wording, or personal usage.
A FAKE review often relies on vague praise, exaggerated enthusiasm, or promotional language without clear evidence of real use.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

prompt_B = """Classify this product review as FAKE or REAL.

Consider whether it reflects genuine product experience or vague promotional language.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

prompt_C = """Classify this product review as FAKE or REAL.

Use cues such as specific experience, vague praise, and promotional wording.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

In [ ]:
prompts = {
    "prompt_A": prompt_A,
    "prompt_B": prompt_B,
    "prompt_C": prompt_C
}

#### 4.2.2. Inference and Output parsing

During inference, each review is embedded into a structured prompt and passed to the model using a chat-based format. The model generates a short textual response, which is constrained to a maximum of a few tokens to ensure concise outputs.

Since the model is generative, the raw output must be processed to extract a valid prediction. Therefore, the generated response is decoded, trimmed, and parsed to obtain a clean label (FAKE or REAL), ensuring consistency across predictions.

In [ ]:
def classify_review(review_text, prompt_template):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a product review classifier. "
                "Classify each review as FAKE or REAL. "
                "Output only one word. "
                "Allowed outputs: FAKE or REAL. "
                "Do not explain. "
                "Do not add punctuation. "
                "Do not add extra text."
            )
        },
        {
            "role": "user",
            "content": prompt_template.format(review=review_text)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    return response.strip()

In [ ]:
def parse_prediction(text):
    if text is None:
        return None

    text = text.strip().upper()

    # Caso exacto ideal
    if text == "FAKE":
        return 1
    if text == "REAL":
        return 0

    # Buscar etiquetas aisladas
    matches = re.findall(r"\b(FAKE|REAL)\b", text)

    if not matches:
        return None

    # Tomar la última, que suele ser la respuesta final
    last_label = matches[-1]

    if last_label == "FAKE":
        return 1
    elif last_label == "REAL":
        return 0

    return None

#### 4.2.3. Prediction Generation

In [ ]:
test_df = pd.DataFrame(ds["test"]).head(100).copy()

In [ ]:
for prompt_name, prompt_template in prompts.items():

    preds = []
    raw_outputs = []

    for review in test_df["text"]:

        output = classify_review(review, prompt_template)

        raw_outputs.append(output)
        preds.append(parse_prediction(output))

    test_df[f"raw_{prompt_name}"] = raw_outputs
    test_df[f"pred_{prompt_name}"] = preds

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

#### 4.2.4. Model evaluation

In [ ]:
for prompt_name in prompts.keys():

    valid = test_df[test_df[f"pred_{prompt_name}"].notna()]

    recall = recall_score(
        valid["label"],
        valid[f"pred_{prompt_name}"],
        pos_label=1,
        zero_division=0
    )

    print(prompt_name, recall)

prompt_A 0.40425531914893614
prompt_B 0.723404255319149
prompt_C 0.7021276595744681


In [ ]:
results = []

for prompt_name in prompts.keys():

    for group in ["short","medium","long"]:

        subset = test_df[
            (test_df["length_group"] == group) &
            (test_df[f"pred_{prompt_name}"].notna())
        ]

        rec = recall_score(
            subset["label"],
            subset[f"pred_{prompt_name}"],
            pos_label=1,
            zero_division=0
        )

        results.append({
            "prompt": prompt_name,
            "length_group": group,
            "n": len(subset),
            "recall": rec
        })

results_df = pd.DataFrame(results)
print(results_df)

     prompt length_group   n    recall
0  prompt_A        short  40  0.350000
1  prompt_A       medium  31  0.437500
2  prompt_A         long  29  0.454545
3  prompt_B        short  40  0.600000
4  prompt_B       medium  31  0.812500
5  prompt_B         long  29  0.818182
6  prompt_C        short  40  0.650000
7  prompt_C       medium  31  0.750000
8  prompt_C         long  29  0.727273


###### 4.2.4.1. Prompt Disagreement

To assess the consistency of the model, we compute the disagreement rate between different prompt formulations. A disagreement occurs when at least one prompt produces a different prediction for the same review.

In [ ]:
test_df["prompt_disagreement"] = (
    (test_df["pred_prompt_A"] != test_df["pred_prompt_B"]) |
    (test_df["pred_prompt_A"] != test_df["pred_prompt_C"]) |
    (test_df["pred_prompt_B"] != test_df["pred_prompt_C"])
)

In [ ]:
change_rate = test_df["prompt_disagreement"].mean()

print("Prompt disagreement rate:", change_rate)

Prompt disagreement rate: 0.34


In [ ]:
test_df.groupby("length_group")["prompt_disagreement"].mean()

,prompt_disagreement
length_group,
long,0.448276
medium,0.354839
short,0.250000


In [ ]:
test_df[test_df["prompt_disagreement"]].head(5)

,text,label,n_tokens,length_group,raw_prompt_A,pred_prompt_A,raw_prompt_B,pred_prompt_B,raw_prompt_C,pred_prompt_C,prompt_disagreement
2,Update 11/28/2015 - Almost 2 years later and t...,0,140,long,REAL,0,FAKE,1,FAKE,1,True
3,"I bought this in November, 2011. It worked fan...",0,170,long,REAL,0,FAKE,1,FAKE,1,True
4,I used to buy and play with only brand-new bal...,0,511,long,REAL,0,FAKE,1,FAKE,1,True
5,"These are for a 13 yr old boy, needs for snow,...",1,38,short,REAL,0,FAKE,1,FAKE,1,True
8,"i really like this ink, it fills in very well ...",0,27,short,REAL,0,FAKE,1,FAKE,1,True


##4.3. Qwen Model

## 5. Comparative Results